# Raw Data EDA Before Preprocessing

This notebook is a compact exploratory notebook for Session 03.

It is intentionally **outside the final data product pipeline**. It is not a preprocessing notebook, not a production artifact, and not a reusable component. The goal is to understand the raw scrape before deciding how preprocessing rules should work.

We will use this notebook to inspect:

- raw dataset structure
- scraping coverage and duplicate behavior
- missingness and suspicious placeholders
- numeric distributions and extreme values
- categorical inconsistencies
- temporal patterns in `registration`
- exploratory quality flags that may justify later preprocessing decisions

We do **not** clean the dataset here, do **not** save transformed outputs, and do **not** produce pipeline artifacts.

In [ ]:
# Imports for a fast exploratory sweep.
from pathlib import Path
import re

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.basedatatypes as pbd
from IPython.display import HTML, Markdown, display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 50)
pd.set_option("display.width", 140)

# Fall back to inline HTML rendering when nbformat is missing in the environment.
try:
    import nbformat  # noqa: F401
except ImportError:
    def _inline_plotly_show(self, *args, **kwargs):
        html = self.to_html(full_html=False, include_plotlyjs="cdn")
        display(HTML(html))

    pbd.BaseFigure.show = _inline_plotly_show

# Resolve the repository root so the notebook works from the repo root or from notebooks/.
cwd = Path.cwd().resolve()
root_candidates = [cwd, cwd.parent, cwd.parent.parent]
repo_root = next((path for path in root_candidates if (path / "data" / "raw").exists()), None)

if repo_root is None:
    raise FileNotFoundError(
        f"Could not locate the repository data folder from working directory: {cwd}"
    )

# Prefer the requested raw file and fall back to the latest matching extract if needed.
raw_dir = repo_root / "data" / "raw"
default_raw_path = raw_dir / "autoscout24_listings_audi_a3_germany_20251228_104706.csv"
fallback_candidates = [path for path in raw_dir.glob("autoscout24_listings_audi_a3_germany_*.csv") if path.stat().st_size > 10]

if default_raw_path.exists():
    raw_path = default_raw_path
    print(f"Using requested file: {raw_path}")
elif fallback_candidates:
    raw_path = max(fallback_candidates, key=lambda path: path.stat().st_mtime)
    print(f"Requested file not found. Falling back to latest matching file: {raw_path}")
else:
    raise FileNotFoundError(
        f"No raw Audi A3 Germany CSV file was found in {raw_dir}."
    )

# Keep loading logic explicit and fail with a readable message.
try:
    df_raw = pd.read_csv(raw_path)
except Exception as exc:
    raise RuntimeError(f"Could not load raw CSV: {raw_path}") from exc

# Create typed views for exploration only. The raw dataframe stays unchanged.
eda_df = df_raw.copy()
eda_df["price_num"] = pd.to_numeric(eda_df["price"], errors="coerce")
eda_df["mileage_num"] = pd.to_numeric(eda_df["mileage"], errors="coerce")
eda_df["power_hp_num"] = pd.to_numeric(eda_df["power_hp"], errors="coerce")

registration_text = eda_df["registration"].astype("string")
registration_is_standard = registration_text.str.match(r"^(0[1-9]|1[0-2])-\d{4}$", na=False)
eda_df["registration_date"] = pd.to_datetime(
    registration_text.where(registration_is_standard),
    format="%m-%Y",
    errors="coerce",
)
eda_df["registration_year"] = eda_df["registration_date"].dt.year

# This subset acts like an approximate listing signature when page is ignored.
listing_signature_cols = [col for col in df_raw.columns if col != "page"]
exact_duplicate_rows = int(df_raw.duplicated().sum())
approx_duplicate_rows = int(df_raw.duplicated(subset=listing_signature_cols).sum())

print(f"Rows: {len(df_raw):,}")
print(f"Columns: {len(df_raw.columns)}")
print(f"Exact duplicate rows: {exact_duplicate_rows:,}")
print(f"Approximate duplicate rows without page: {approx_duplicate_rows:,}")

## 1. Load and Quick Inspection

This is only a first contact with the raw extract. Before defining any rule, we want to see what columns exist, how values look, and which fields already hint at possible quality questions.

In [ ]:
# Show the basic shape, columns, dtypes, and a few sample rows.
overview_df = pd.DataFrame(
    {
        "metric": ["rows", "columns", "file_name"],
        "value": [len(df_raw), len(df_raw.columns), raw_path.name],
    }
)
display(overview_df)

display(pd.DataFrame({"column": df_raw.columns, "raw_dtype": df_raw.dtypes.astype(str).values}))
display(df_raw.head(10))

## 2. Dataset Health Snapshot

The point of this section is speed. We want a compact overview of raw dataset reliability and structure before diving into details.

Metric guide:

- `rows`: total number of raw observations loaded from the file.
- `columns`: total number of raw fields available in the extract.
- `pages`: number of distinct scraping result pages represented in the dataset.
- `exact_duplicate_rows`: rows that are identical across every column, including `page`.
- `approx_duplicate_rows_without_page`: rows that become duplicates when `page` is ignored. In this notebook, the matching criteria is: all columns except `page` have the same value. This is useful because the same listing may appear on multiple pages even when it is not literally an exact row duplicate.
- `non_standard_registration_rows`: rows where `registration` does not match the expected `MM-YYYY` pattern used in this notebook for safe parsing.
- `non_numeric_mileage_rows`: rows where `mileage` cannot be converted to a numeric value with `pd.to_numeric(..., errors='coerce')`.

How the `suspicious_rows` column is built in the profile table:

- For every column, a row is marked as suspicious if the raw value is missing, blank after trimming, or equal to placeholders such as `unknown`, `none`, or `nan`.
- For numeric columns (`price`, `mileage`, `power_hp`), a row is also suspicious if the raw value exists but cannot be converted to a number.
- For `registration`, a row is also suspicious if it does not match the regex `^(0[1-9]|1[0-2])-\d{4}$`.
- This is an exploratory indicator, not a final quality verdict.

In [ ]:
# Build a simple suspicious-value profile by column.
suspicious_rows = []

for column in df_raw.columns:
    series = df_raw[column]
    text_series = series.astype("string")

    suspicious_mask = series.isna()
    suspicious_mask |= text_series.str.strip().eq("").fillna(False)
    suspicious_mask |= text_series.str.strip().str.lower().isin(["unknown", "none", "nan"]).fillna(False)

    if column in ["price", "mileage", "power_hp"]:
        suspicious_mask |= pd.to_numeric(series, errors="coerce").isna() & series.notna()

    if column == "registration":
        suspicious_mask |= ~text_series.str.match(r"^(0[1-9]|1[0-2])-\d{4}$", na=False)

    suspicious_rows.append(
        {
            "column": column,
            "suspicious_rows": int(suspicious_mask.sum()),
            "suspicious_share_pct": round(100 * suspicious_mask.mean(), 2),
            "unique_values": int(text_series.nunique(dropna=False)),
        }
    )

suspicious_profile = pd.DataFrame(suspicious_rows).sort_values(
    ["suspicious_rows", "unique_values"], ascending=[False, False]
)

snapshot_df = pd.DataFrame(
    [
        {"metric": "rows", "value": len(df_raw)},
        {"metric": "columns", "value": len(df_raw.columns)},
        {"metric": "pages", "value": int(df_raw["page"].nunique())},
        {"metric": "exact_duplicate_rows", "value": exact_duplicate_rows},
        {"metric": "approx_duplicate_rows_without_page", "value": approx_duplicate_rows},
        {"metric": "non_standard_registration_rows", "value": int(eda_df["registration_year"].isna().sum())},
        {"metric": "non_numeric_mileage_rows", "value": int(eda_df["mileage_num"].isna().sum())},
    ]
)

display(snapshot_df)
display(suspicious_profile)

# Records by page help us check whether scraping coverage is flat or irregular.
page_counts = (
    df_raw["page"]
    .value_counts()
    .sort_index()
    .rename_axis("page")
    .reset_index(name="records")
)
fig = px.bar(
    page_counts,
    x="page",
    y="records",
    title="Records by page",
    labels={"page": "Page", "records": "Rows"},
)
fig.update_layout(height=350)
fig.show()

# Suspicious counts by column highlight where raw issues concentrate.
fig = px.bar(
    suspicious_profile.sort_values("suspicious_rows", ascending=False),
    x="column",
    y="suspicious_rows",
    text="suspicious_rows",
    title="Missing or suspicious values by column",
    hover_data=["suspicious_share_pct", "unique_values"],
)
fig.update_layout(height=350)
fig.show()

# Cardinality keeps the discussion grounded in the main categorical fields.
key_categorical_cols = ["make", "model", "price_label", "fuel", "country", "country_name"]
cardinality_df = pd.DataFrame(
    {
        "column": key_categorical_cols,
        "unique_values": [df_raw[col].astype("string").nunique(dropna=False) for col in key_categorical_cols],
    }
)
fig = px.bar(
    cardinality_df,
    x="column",
    y="unique_values",
    text="unique_values",
    title="Cardinality of key categorical columns",
)
fig.update_layout(height=350)
fig.show()

## 3. Scraping Coverage and Duplication

Some quality issues come from how the data was collected. Repeated listings across pages can create duplicate-like patterns that are not really business anomalies.

Interpretation guide:

- A `listing signature` in this notebook means all raw columns except `page`.
- `unique listing signatures` counts distinct combinations of those fields after ignoring `page`.
- `repeated listing signatures` counts how many of those signatures appear more than once.
- `rows in repeated signatures` counts all rows that belong to repeated signatures, not just the extra copies.
- `max pages for one signature` shows the largest number of distinct pages on which the same signature appears.

This is why `approx_duplicate_rows_without_page` is useful: it helps distinguish repeated acquisition from true entity-level duplication problems.

In [ ]:
# Count repeated listing signatures after ignoring page.
duplicate_groups = (
    df_raw.groupby(listing_signature_cols, dropna=False)
    .agg(
        occurrences=("page", "size"),
        distinct_pages=("page", "nunique"),
        first_page=("page", "min"),
        last_page=("page", "max"),
    )
    .reset_index()
    .sort_values(["occurrences", "distinct_pages"], ascending=False)
)

repeated_signatures = duplicate_groups[duplicate_groups["occurrences"] > 1].copy()

duplicate_pattern_df = pd.DataFrame(
    [
        {"metric": "unique listing signatures", "value": int(duplicate_groups.shape[0])},
        {"metric": "repeated listing signatures", "value": int(repeated_signatures.shape[0])},
        {"metric": "rows in repeated signatures", "value": int(repeated_signatures["occurrences"].sum()) if not repeated_signatures.empty else 0},
        {"metric": "max pages for one signature", "value": int(repeated_signatures["distinct_pages"].max()) if not repeated_signatures.empty else 1},
    ]
)

display(duplicate_pattern_df)
display(
    repeated_signatures[
        ["price", "mileage", "registration", "fuel", "power_hp", "price_label", "occurrences", "distinct_pages", "first_page", "last_page"]
    ].head(15)
)

# A histogram of repeated signatures shows whether duplication is rare or systematic.
if not repeated_signatures.empty:
    fig = px.histogram(
        repeated_signatures,
        x="occurrences",
        nbins=10,
        title="How many times does the same listing signature appear?",
        labels={"occurrences": "Occurrences across rows"},
    )
    fig.update_layout(height=350)
    fig.show()

## 4. Numeric Distributions

Before calling anything an outlier, we need to see how the market behaves. Used-car variables are usually skewed, and long tails may reflect real subsegments rather than obvious errors.

In [ ]:
# Histograms show the main shape of the numeric variables.
fig = px.histogram(
    eda_df,
    x="price_num",
    nbins=60,
    title="Price distribution",
    labels={"price_num": "Price (EUR)"},
)
fig.update_layout(height=350)
fig.show()

fig = px.histogram(
    eda_df,
    x="mileage_num",
    nbins=60,
    title="Mileage distribution",
    labels={"mileage_num": "Mileage (km)"},
)
fig.update_layout(height=350)
fig.show()

fig = px.histogram(
    eda_df,
    x="power_hp_num",
    nbins=50,
    title="Power distribution",
    labels={"power_hp_num": "Power (HP)"},
)
fig.update_layout(height=350)
fig.show()

In [ ]:
# A direct log x-axis can render poorly in some notebook environments.
# For teaching, a histogram of log10 values is more stable and easier to read.
positive_price_df = eda_df[eda_df["price_num"] > 0].copy()
positive_price_df["log10_price"] = np.log10(positive_price_df["price_num"])

fig = px.histogram(
    positive_price_df,
    x="log10_price",
    nbins=40,
    title="Price distribution (log10 scale)",
    labels={"log10_price": "log10(Price in EUR)"},
)
fig.update_xaxes(
    tickvals=[2, 3, 4, 5],
    ticktext=["100", "1k", "10k", "100k"],
)
fig.update_layout(height=350)
fig.show()

positive_mileage_df = eda_df[eda_df["mileage_num"] > 0].copy()
positive_mileage_df["log10_mileage"] = np.log10(positive_mileage_df["mileage_num"])

fig = px.histogram(
    positive_mileage_df,
    x="log10_mileage",
    nbins=40,
    title="Mileage distribution (log10 scale)",
    labels={"log10_mileage": "log10(Mileage in km)"},
)
fig.update_xaxes(
    tickvals=[1, 2, 3, 4, 5, 6],
    ticktext=["10", "100", "1k", "10k", "100k", "1M"],
)
fig.update_layout(height=350)
fig.show()

## 5. Numeric Anomaly Views

Boxplots are useful here because they force a discussion: is an extreme value an error, a rare car, or a different market segment?

In [ ]:
# Segment the raw values by categories that already exist in the extract.
fig = px.box(
    eda_df,
    x="price_label",
    y="price_num",
    points="suspectedoutliers",
    title="Price by raw price label",
    labels={"price_label": "Raw price label", "price_num": "Price (EUR)"},
)
fig.update_layout(height=400)
fig.show()

fig = px.box(
    eda_df,
    x="fuel",
    y="mileage_num",
    points="suspectedoutliers",
    title="Mileage by raw fuel code",
    labels={"fuel": "Raw fuel", "mileage_num": "Mileage (km)"},
)
fig.update_layout(height=400)
fig.show()

fig = px.box(
    eda_df,
    x="fuel",
    y="power_hp_num",
    points="suspectedoutliers",
    title="Power by raw fuel code",
    labels={"fuel": "Raw fuel", "power_hp_num": "Power (HP)"},
)
fig.update_layout(height=400)
fig.show()

## 6. Key Bivariate Relationships

If the data roughly behaves like a used-car market, relationships such as higher mileage to lower price should start to appear. Strange clusters deserve interpretation before they become automatic preprocessing rules.

In [ ]:
# Scatter plots help us check whether the economics look plausible.
fig = px.scatter(
    eda_df,
    x="mileage_num",
    y="price_num",
    color="price_label",
    opacity=0.55,
    title="Price vs mileage",
    labels={"mileage_num": "Mileage (km)", "price_num": "Price (EUR)"},
    hover_data=["registration", "fuel", "power_hp", "page"],
)
fig.update_layout(height=450)
fig.show()

fig = px.scatter(
    eda_df.dropna(subset=["registration_year"]),
    x="registration_year",
    y="price_num",
    color="fuel",
    opacity=0.55,
    title="Price vs registration year",
    labels={"registration_year": "Registration year", "price_num": "Price (EUR)"},
    hover_data=["registration", "mileage", "power_hp", "page"],
)
fig.update_layout(height=450)
fig.show()

fig = px.scatter(
    eda_df,
    x="power_hp_num",
    y="price_num",
    color="fuel",
    opacity=0.55,
    title="Price vs power",
    labels={"power_hp_num": "Power (HP)", "price_num": "Price (EUR)"},
    hover_data=["registration", "mileage", "price_label", "page"],
)
fig.update_layout(height=450)
fig.show()

## 7. Categorical Variable Review

Raw categories often reveal standardization work that comes later. At this stage we want to see the categories exactly as they arrived.

In [ ]:
# Frequency charts make raw category issues visible very quickly.
price_label_counts = (
    df_raw["price_label"]
    .astype("string")
    .value_counts(dropna=False)
    .rename_axis("price_label")
    .reset_index(name="rows")
)
fig = px.bar(
    price_label_counts,
    x="price_label",
    y="rows",
    text="rows",
    title="Raw price_label frequencies",
)
fig.update_layout(height=350)
fig.show()

fuel_counts = (
    df_raw["fuel"]
    .astype("string")
    .value_counts(dropna=False)
    .rename_axis("fuel")
    .reset_index(name="rows")
)
fig = px.bar(
    fuel_counts,
    x="fuel",
    y="rows",
    text="rows",
    title="Raw fuel frequencies",
)
fig.update_layout(height=350)
fig.show()

country_counts = (
    df_raw["country"]
    .astype("string")
    .value_counts(dropna=False)
    .rename_axis("country")
    .reset_index(name="rows")
)
fig = px.bar(
    country_counts,
    x="country",
    y="rows",
    text="rows",
    title="Raw country frequencies",
)
fig.update_layout(height=300)
fig.show()

country_name_counts = (
    df_raw["country_name"]
    .astype("string")
    .value_counts(dropna=False)
    .rename_axis("country_name")
    .reset_index(name="rows")
)
fig = px.bar(
    country_name_counts,
    x="country_name",
    y="rows",
    text="rows",
    title="Raw country_name frequencies",
)
fig.update_layout(height=300)
fig.show()

In [ ]:
# Show concrete examples of raw categories that deserve discussion later.
category_notes = pd.DataFrame(
    [
        {
            "field": "price_label",
            "observation": "At least one category contains trailing whitespace.",
            "example": next(
                (value for value in df_raw["price_label"].astype("string").dropna().unique() if value != value.strip()),
                "not found",
            ),
        },
        {
            "field": "fuel",
            "observation": "Fuel values look like short raw codes, not classroom-ready labels.",
            "example": ", ".join(sorted(df_raw["fuel"].astype("string").dropna().unique().tolist())),
        },
        {
            "field": "country",
            "observation": "The code column uses a short code while country_name keeps a readable label.",
            "example": f"country={df_raw['country'].iloc[0]!r}, country_name={df_raw['country_name'].iloc[0]!r}",
        },
    ]
)
display(category_notes)

## 8. Registration and Temporal Structure

Date fields are not only a parsing problem. Here we want to see how many values look standard, how many are special cases, and what the registration timeline suggests about the market.

In [ ]:
# Parse only the values that match the expected month-year format.
registration_year_counts = (
    eda_df["registration_year"]
    .dropna()
    .astype(int)
    .value_counts()
    .sort_index()
    .rename_axis("registration_year")
    .reset_index(name="rows")
)

fig = px.bar(
    registration_year_counts,
    x="registration_year",
    y="rows",
    title="Registration year distribution (parsed values only)",
    labels={"registration_year": "Registration year", "rows": "Rows"},
)
fig.update_layout(height=350)
fig.show()

registration_text = df_raw["registration"].astype("string")
problematic_registration = (
    registration_text[~registration_text.str.match(r"^(0[1-9]|1[0-2])-\d{4}$", na=False)]
    .value_counts(dropna=False)
    .rename_axis("raw_registration")
    .reset_index(name="rows")
)

display(problematic_registration)

## 9. Data Quality Rule Exploration

These are exploratory flags, not final preprocessing rules. The point is to turn raw observations into hypotheses about what later cleaning may need to examine.

Flag guide used in this notebook:

- `zero_or_near_zero_mileage`: `mileage_num` is between `0` and `100` km inclusive. This is not automatically wrong, but it deserves attention because some rows may represent brand-new cars, placeholders, or unusual listings.
- `extreme_mileage_over_250k`: `mileage_num` is greater than `250,000` km.
- `extreme_low_price_under_1000`: `price_num` is lower than `1,000` EUR.
- `missing_or_non_numeric_power`: `power_hp` is missing or cannot be converted to numeric.
- `non_standard_registration`: `registration` could not be parsed with the notebook's safe `MM-YYYY` rule.
- `very_new_with_very_high_mileage`: parsed `registration_year` is `2023` or later and `mileage_num` is above `120,000` km.
- `very_old_with_near_zero_mileage`: parsed `registration_year` is `2008` or earlier and `mileage_num` is between `0` and `100` km inclusive.

Important: these thresholds are classroom discussion thresholds only. They are meant to generate questions before preprocessing, not to hard-code final cleaning decisions.

In [ ]:
# Create discussion flags without filtering or saving a cleaned dataset.
flags_df = pd.DataFrame(index=eda_df.index)
flags_df["zero_or_near_zero_mileage"] = eda_df["mileage_num"].fillna(-1).between(0, 100)
flags_df["extreme_mileage_over_250k"] = eda_df["mileage_num"] > 250_000
flags_df["extreme_low_price_under_1000"] = eda_df["price_num"] < 1_000
flags_df["missing_or_non_numeric_power"] = eda_df["power_hp_num"].isna()
flags_df["non_standard_registration"] = eda_df["registration_year"].isna()
flags_df["very_new_with_very_high_mileage"] = (eda_df["registration_year"] >= 2023) & (eda_df["mileage_num"] > 120_000)
flags_df["very_old_with_near_zero_mileage"] = (eda_df["registration_year"] <= 2008) & (eda_df["mileage_num"].fillna(-1).between(0, 100))

flag_counts = (
    flags_df.sum()
    .sort_values(ascending=False)
    .rename_axis("flag")
    .reset_index(name="rows")
)
flag_counts["share_pct"] = (100 * flag_counts["rows"] / len(df_raw)).round(2)

display(flag_counts)

fig = px.bar(
    flag_counts,
    x="rows",
    y="flag",
    orientation="h",
    text="rows",
    title="Exploratory quality flags",
    hover_data=["share_pct"],
)
fig.update_layout(height=400)
fig.show()

# Show a few examples so the class can inspect them instead of treating flags as automatic errors.
any_flag_mask = flags_df.any(axis=1)
flagged_examples = pd.concat([eda_df.loc[any_flag_mask, ["price", "mileage", "registration", "fuel", "power_hp", "price_label", "page"]], flags_df.loc[any_flag_mask]], axis=1)
display(flagged_examples.head(15))

## 10. Segment-Quality Cross Views

Quality issues are not always evenly distributed. Sometimes they cluster in particular segments, which matters when later preprocessing decisions are designed.

In [ ]:
# Cross tabulate raw categories to see whether some segments concentrate certain labels.
fuel_price_label_ct = pd.crosstab(df_raw["fuel"], df_raw["price_label"])
fig = go.Figure(
    data=go.Heatmap(
        z=fuel_price_label_ct.values,
        x=fuel_price_label_ct.columns,
        y=fuel_price_label_ct.index,
        colorscale="Blues",
        text=fuel_price_label_ct.values,
        texttemplate="%{text}",
    )
)
fig.update_layout(
    title="Fuel x price_label",
    xaxis_title="Raw price_label",
    yaxis_title="Raw fuel",
    height=380,
)
fig.show()

# Bucket the parsed years to keep the temporal segment discussion simple.
year_bucket = pd.Series("missing/non-standard", index=eda_df.index)
year_bucket = year_bucket.mask(eda_df["registration_year"].between(1996, 2009), "< 2010")
year_bucket = year_bucket.mask(eda_df["registration_year"].between(2010, 2014), "2010-2014")
year_bucket = year_bucket.mask(eda_df["registration_year"].between(2015, 2019), "2015-2019")
year_bucket = year_bucket.mask(eda_df["registration_year"].between(2020, 2022), "2020-2022")
year_bucket = year_bucket.mask(eda_df["registration_year"] >= 2023, "2023+")

year_price_label_ct = (
    pd.crosstab(year_bucket, df_raw["price_label"])
    .reindex(["< 2010", "2010-2014", "2015-2019", "2020-2022", "2023+", "missing/non-standard"])
    .fillna(0)
)

year_price_label_long = (
    year_price_label_ct
    .reset_index(names="registration_year_bucket")
    .melt(id_vars="registration_year_bucket", var_name="price_label", value_name="rows")
)
fig = px.bar(
    year_price_label_long,
    x="registration_year_bucket",
    y="rows",
    color="price_label",
    barmode="stack",
    title="Registration year bucket x price_label",
    labels={"registration_year_bucket": "Registration year bucket"},
)
fig.update_layout(height=400)
fig.show()

## What We Learned Before Preprocessing

The value of this notebook is not to clean the data. It is to make later preprocessing decisions more defensible.

In [ ]:
# Summarize the main takeaways with dataset-specific evidence.
repeated_signature_count = int((duplicate_groups["occurrences"] > 1).sum())
non_standard_registration_count = int(eda_df["registration_year"].isna().sum())
non_numeric_mileage_count = int(eda_df["mileage_num"].isna().sum())
zero_or_near_zero_mileage_count = int(flags_df["zero_or_near_zero_mileage"].sum())

summary_md = f'''
- The raw extract contains **{len(df_raw):,} rows** across **{int(df_raw['page'].nunique())} pages**, so coverage is broad enough to expose scraping behavior.
- We found **{exact_duplicate_rows:,} exact duplicates** and **{approx_duplicate_rows:,} approximate duplicates when page is ignored**, which suggests that repeated listings can come from acquisition logic rather than business reality.
- `registration` is not purely a date field: **{non_standard_registration_count:,} rows** do not match the expected `MM-YYYY` pattern, and values such as `new` need interpretation before parsing rules are fixed.
- Numeric fields are not clean by default. For example, **{non_numeric_mileage_count:,} mileage values** are not numeric, and **{zero_or_near_zero_mileage_count:,} rows** have zero or near-zero mileage.
- Raw categories already show standardization work ahead: `fuel` uses short codes, `country` and `country_name` carry the same idea in different forms, and `price_label` includes at least one whitespace issue.
- Price, mileage, and power are clearly skewed, so simple visual inspection is necessary before using any outlier rule.
- Some suspicious combinations may be real edge cases rather than clear errors, which is why later preprocessing should separate **raw anomaly**, **possible data issue**, and **market niche**.
- This notebook stops at explanation. No cleaned dataframe is produced here, and no transformed file is saved.
'''

display(Markdown(summary_md))